# Notebook 01 — Datos semiestructurados: hstore y JSONB

Tema 09. PostgreSQL permite columnas cuyo schema **no es fijo**: una sola columna guarda pares clave-valor (`hstore`) o estructuras JSON anidadas (`JSONB`). Resuelve el problema de entidades heterogéneas con atributos distintos por instancia.

Cubres: `hstore` (operadores `->`, `||`, `?` e indexación GIN) y `JSON`/`JSONB` (operadores `->`, `->>`, `#>>`, `@>`, las funciones `jsonb_array_elements()` y `jsonb_extract_path()`, e indexación GIN/btree).

**Contenido de este notebook:**

- [Setup](#setup)
- [¿Por qué datos semiestructurados?](#por-qué-datos-semiestructurados)
- [`hstore` — clave-valor plano](#hstore--clave-valor-plano)
- [`JSONB` — estructuras jerárquicas](#jsonb--estructuras-jerárquicas)
- [JSONB sobre datos reales: `amenities` de Airbnb](#jsonb-sobre-datos-reales-amenities-de-airbnb)
- [Indexación: GIN y btree](#indexación-gin-y-btree)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## ¿Por qué datos semiestructurados?

El problema: atributos que **cambian por entidad**. Un libro tiene `paginas` y `autor`; una camiseta tiene `talla` y `color`. Modelar cada variante como columna propia explota la tabla de nulos. PostgreSQL ofrece guardar la estructura variable **dentro de una columna**:

- **`hstore`** — colección **plana** de pares clave-valor (todo texto). Simple y rápido, pero sin anidamiento.
- **`json`** — JSON **jerárquico** guardado como **texto tal cual** lo escribiste (objetos, arrays, anidamiento), pero sin indexar ni normalizar.
- **`JSONB`** — el mismo JSON pero en formato **binario descompuesto**: PostgreSQL lo parsea al insertar, lo que permite operadores e índices (GIN) y consultas rápidas.

### `json` vs `jsonb`

Los dos guardan JSON, pero `json` conserva el **texto literal** y `jsonb` lo **descompone en estructura binaria** al insertar. Se ve en la salida (corre la celda de abajo para comprobarlo):

```sql
SELECT '{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}'::json  AS texto,
       '{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}'::jsonb AS descompuesto;
```

- `json`  → `{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}`  &larr; idéntico, conserva los espacios extra
- `jsonb` → `{"host": {"name": "Ana"}, "wifi": true, "precio": 850}`  &larr; sin espacios y con las claves reordenadas

Ese cambio en la salida de `jsonb` es la evidencia de que ya no se guarda como texto. Cuatro consecuencias prácticas:

1. **Formato textual** — `json` lo preserva exacto; `jsonb` normaliza (se pierden los espacios y el formato original).
2. **Orden de claves** — `json` lo respeta; `jsonb` reordena internamente **por longitud primero y luego alfabético** (por eso `host` y `wifi`, de 4 letras, van antes que `precio`, de 6).
3. **Claves duplicadas** — `json` las conserva todas; `jsonb` se queda solo con la última.
4. **Índices y operadores** — solo `jsonb` admite índices **GIN** y operadores como `@>` (containment); `json` no.

> **Ojo con el visor:** clientes como DBeaver o Colab detectan un valor `json`/`jsonb` y lo **reformatean** al pintarlo, así que en la grilla parecería que `json` también pierde los espacios. Por eso la celda castea a **`::text`** (el cliente lo trata como cadena normal y ya no lo embellece) y compara `length()`: ahí se ve que `json` conserva los espacios (`len_json` = 55) y `jsonb` no (`len_jsonb` = 54).

**Regla práctica:** usa **`jsonb`** salvo que necesites preservar el texto original byte por byte (p. ej. auditar un payload tal como llegó de una API).

In [ ]:
%%sql
-- json conserva el texto literal; jsonb lo descompone y normaliza.
-- Casteamos a ::text para que el cliente NO reformatee el JSON al mostrarlo
-- (DBeaver/Colab "embellecen" un valor json y ocultarían los espacios).
SELECT
    '{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}'::json::text         AS texto,
    '{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}'::jsonb::text        AS descompuesto,
    length('{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}'::json::text)  AS len_json,   -- conserva los espacios
    length('{"precio": 850,   "wifi": true, "host": {"name":"Ana"}}'::jsonb::text) AS len_jsonb;  -- normalizado: 1 menos

## `hstore` — clave-valor plano

`hstore` guarda pares `clave => valor` en una sola columna; **claves y valores son texto**. Primero hay que habilitar la extensión (una vez por base):

In [ ]:
%%sql
CREATE EXTENSION IF NOT EXISTS hstore;

Operadores principales: `->` (valor de una clave, **ya como texto**), `?` (¿existe la clave?), `||` (merge de dos hstore):

In [ ]:
%%sql
SELECT
    'talla=>M, color=>azul, material=>algodón'::hstore        AS atributos,
    ('talla=>M, color=>azul'::hstore) -> 'color'              AS color,
    ('talla=>M, color=>azul'::hstore) ? 'material'            AS tiene_material,
    ('talla=>M'::hstore || 'color=>rojo'::hstore)             AS merge;

Nota: en `hstore`, `->` **ya devuelve texto** (todo en hstore es texto), por eso no existe un `->>` aparte como en JSON. La limitación clave de `hstore` es que es **plano**: no admite objetos anidados ni arrays. Para eso, `JSONB`.

## `JSONB` — estructuras jerárquicas

`JSON` guarda el texto tal cual; `JSONB` lo guarda en **binario** (más rápido de consultar e indexable). Regla práctica: usa **`JSONB`** salvo que necesites preservar el formato textual exacto.

Operadores de acceso:

| Operador | Devuelve | Para qué |
|---|---|---|
| `->` | `JSONB` | acceder a una clave/índice (sigue siendo JSON) |
| `->>` | texto | acceder a una clave/índice como texto |
| `#>>` | texto | acceder por **path** a algo anidado |
| `@>` | booleano | ¿**contiene** este fragmento? (clave para índices) |

In [ ]:
%%sql
SELECT
    ('{"host": {"name": "Ana"}, "precio": 850}'::jsonb) -> 'precio'       AS precio_jsonb,
    ('{"host": {"name": "Ana"}, "precio": 850}'::jsonb) ->> 'precio'      AS precio_texto,
    ('{"host": {"name": "Ana"}, "precio": 850}'::jsonb) #>> '{host,name}' AS host_nombre,
    ('{"host": {"name": "Ana"}, "precio": 850}'::jsonb) @> '{"precio": 850}'::jsonb AS contiene_precio;

`->` devuelve `JSONB` (encadenable: `data -> 'host' -> 'name'`); `->>` lo devuelve como **texto** (para comparar o castear); `#>>` baja por un **path** `{host,name}` en un paso. La función `jsonb_extract_path(data, 'host', 'name')` hace lo mismo que `#>` en forma de función.

## JSONB sobre datos reales: `amenities` de Airbnb

La columna `airbnb.listings.amenities` es un **array JSON guardado como texto** (`["Wifi", "Kitchen", …]`). La casteamos a `jsonb` para operar sobre ella. El operador `@>` (containment) responde "¿este listing tiene tal amenity?":

In [ ]:
%%sql
-- ¿Cuántos listings ofrecen Wifi? (containment sobre el array)
SELECT COUNT(*) AS con_wifi
FROM   airbnb.listings
WHERE  amenities::jsonb @> '["Wifi"]'::jsonb;

La función **`jsonb_array_elements()`** (y su variante `_text`) **explota** un array JSON a filas — una fila por elemento. Combinado con `GROUP BY`, da el ranking de amenities más comunes:

In [ ]:
%%sql
SELECT  elem AS amenity, COUNT(*) AS n
FROM    airbnb.listings l,
        LATERAL jsonb_array_elements_text(l.amenities::jsonb) AS elem
WHERE   l.amenities IS NOT NULL AND l.amenities <> '[]'
GROUP   BY elem
ORDER   BY n DESC
LIMIT   10;

`jsonb_array_elements(x)` devuelve los elementos como `JSONB` (con comillas); `jsonb_array_elements_text(x)` los devuelve como texto plano — más cómodo para agrupar. El `LATERAL` permite que la función se aplique fila por fila del `FROM`.

## Indexación: GIN y btree

Sin índice, `@>` recorre toda la tabla. Un índice **GIN** (*Generalized Inverted Index*) indexa los elementos internos del JSON, así que `@>` y `?` se vuelven rápidos. Es el índice natural para semiestructurados:

In [ ]:
%%sql
CREATE INDEX IF NOT EXISTS idx_listings_amenities_gin
    ON airbnb.listings USING GIN ((amenities::jsonb));

In [ ]:
%%sql
EXPLAIN
SELECT COUNT(*) FROM airbnb.listings
WHERE  amenities::jsonb @> '["Pool"]'::jsonb;

El `EXPLAIN` muestra si el planner usa el índice GIN (en tablas grandes lo hace; con pocas filas a veces prefiere el *seq scan*, que igual es barato). 

**GIN vs btree:** GIN sirve para preguntar por el **contenido** de toda la estructura (`@>`, `?`). Un índice **btree** sirve cuando filtras por un **campo escalar específico** con `=`/`<`/`>`, indexando la expresión extraída — p. ej. `CREATE INDEX ON t ((data->>'ciudad'))` para `WHERE data->>'ciudad' = 'CDMX'`.

## Cierre

Lo que cubriste:

| Tipo | Operadores / funciones | Índice |
|---|---|---|
| `hstore` (plano) | `->`, `?`, `\|\|` | GIN |
| `JSONB` (jerárquico) | `->`, `->>`, `#>>`, `@>` | GIN (`@>`,`?`) / btree (campo escalar) |
| Explotar arrays | `jsonb_array_elements()` / `_text` | — |
| Acceso por path (función) | `jsonb_extract_path()` | — |

Con esto cierras el **Módulo 4**: de OLAP y modelado dimensional, pasando por ETL en Python y SQL avanzado, hasta datos semiestructurados. El siguiente paso es el **proyecto final** (ver la rúbrica en `anexos/`).

---

<p align="center">
<a href="../Tema-08/Readme.md">← Anterior: Tema 08</a> | <a href="Readme.md">Volver al índice</a> | <a href="../anexos/rubrica_proyecto_final.md">Siguiente: Proyecto final →</a>
</p>